In [64]:
import pandas as pd

# The Socio-Economic Impact on Education: Predicting State Maturity Exam Scores using Machine Learning

## 1. Problem Statement and Motivation

**Introduction and Context**
Educational outcomes are often viewed as a reflection of a school's quality, but they are deeply intertwined with the broader socio-economic reality of the region. In Bulgaria, the State Maturity Exams (Държавен зрелостен изпит or DZI) serve as the ultimate benchmark for high school education. However, analyzing these grades in a vacuum ignores the macro-economic factors that influence student success, such as regional income disparities, labor market dynamics, and overall living conditions.

**Problem Formulation**
The core objective of this project is to build a Machine Learning regression model that predicts a school's average DZI performance by bridging the gap between micro-level and macro-level data. Specifically, I will investigate whether combining school-specific characteristics (e.g., school profile, cohort size, and historical performance) with district-level socio-economic indicators (e.g., regional income, youth unemployment, and poverty lines) can create a robust predictive model. 

**Project Significance and Objectives**
This approach transforms a simple grade-prediction task into a deeper socio-economic analysis. The primary goals of this project are:
1. **Data Integration:** To successfully merge high-dimensional macro-economic datasets (Regional Profiles) with granular, micro-level educational data (DZI results).
2. **Feature Engineering:** To extract meaningful variables from raw data, such as categorizing school types and utilizing time-series lagged variables for historical performance.
3. **Predictive Modeling:** To train, test, and evaluate various regression algorithms (e.g., Linear Regression, Random Forests) to predict future DZI outcomes.
4. **Interpretability:** To analyze feature importance and understand which socio-economic or school-specific factors carry the most weight in determining educational success.

## 2. Data Loading and Preparation

In [71]:
# 1. Load and combine all DZI datasets (2018-2026)
years = range(2018, 2027)
dzi_frames = []

for year in years:
    yearly_data = pd.read_csv(f'data/dzi-{year}.csv')
    yearly_data['Year'] = year  
    dzi_frames.append(yearly_data)

# Stack them all vertically into one DataFrame
dzi_full = pd.concat(dzi_frames, ignore_index=True)

# 2. Load the Macro-Economic Datasets 
income_data = pd.read_csv('data/E1_25.csv')
labor_data = pd.read_csv('data/E2_25.csv')
edu_data = pd.read_csv('data/S2_25.csv')

In [91]:
print(dzi_full.shape)
dzi_full.head(3)

(8675, 240)


,﻿Област,Община,Населено място,Код по Админ,Училище,БЕЛ Брой,БЕЛ Ср.успех,МАТ Брой,МАТ Ср.успех,ИЦ Брой,...,Бр. ИтЕ(ПП) B2-З,Ср.усп. ИтЕ(ПП) B2-З,Бр. ДИППК-п.р З,Ср.усп. ДИППК-п.р З,Бр. ДИППК-тест З,Ср.усп. ДИППК-тест З,Бр. ДИППК-Д.Пр З,Ср.усп. ДИППК-Д.Пр З,Бр. ДИППК-пр. З,Ср.усп. ДИППК-пр. З
0,БЛАГОЕВГРАД,БАНСКО,ГР.БАНСКО,102015,Професионална гимназия по електроника и енерге...,38.0,"4,048",0.0,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,БЛАГОЕВГРАД,БАНСКО,ГР.БАНСКО,102004,"Професионална лесотехническа гимназия""Никола Й...",39.0,"3,916",0.0,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,БЛАГОЕВГРАД,БАНСКО,ГР.БАНСКО,102010,"Гимназия по туризъм ""Алеко Константинов""",18.0,"3,454",0.0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
print(income_data.shape)
income_data.head()

(32, 55)


,Наименование,"БВП на човек от населението, лв.",Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,"Относителен дял на населението, живеещо под линията на бедността за страната, %",Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54
0,Година,2012.0,2013.0,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,...,2021.0,2022.0,2023,2024.0,2019.0,2020.0,2021.0,2022,2023.0,2024.0
1,Благоевград,7533.0,7660.0,7659.0,8013.0,8449.0,9069.0,9989.0,10627.0,11179.0,...,34.1,32.1,27.6,28.6,23.9,25.1,19.1,22.9,21.6,19.5
2,Бургас,9881.0,10101.0,9449.0,10943.0,12174.0,13283.0,13460.0,14483.0,12351.0,...,32.6,38.0,29.7,31.7,20.0,26.5,24.6,22.1,21.0,23.0
3,Варна,11631.0,11543.0,12576.0,13285.0,13756.0,14992.0,16537.0,17545.0,17379.0,...,35.9,33.5,33.1,37.1,18.4,22.9,17.3,14.2,14.6,15.2
4,Велико Търново,7586.0,7974.0,8075.0,8665.0,9121.0,9950.0,11262.0,12078.0,12448.0,...,32.1,30.8,32.8,27.3,25.8,30.8,22.5,20.7,28.9,25.7


In [89]:
print(labor_data.shape)
labor_data.head()

(32, 68)


,Наименование,Относителен дял на населениeто на възраст между 25 и 64 навършени години с висше образование (%),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 58,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,Unnamed: 63,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67
0,Година,2012.0,2013.0,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023,2024.0
1,Благоевград,17.7,18.0,19.6,19.5,19.4,19.9,20.7,21.8,21.1,...,63.0,62.6,62.1,61.7,61.2,61.1,61.0,59.8,59.5,59.2
2,Бургас,18.6,20.2,18.8,19.3,23.1,24.8,23.6,22.5,24.2,...,61.6,61.4,60.9,60.5,60.2,60.1,60.2,58.9,58.7,58.7
3,Варна,26.0,31.4,33.8,30.6,29.9,32.5,29.5,25.3,24.8,...,62.6,62.4,62.1,61.8,61.7,61.7,62.0,60.3,60.4,60.5
4,Велико Търново,23.5,26.6,27.3,26.9,27.3,22.1,26.8,29.6,29.7,...,60.1,59.8,59.5,59.3,59.1,59.1,59.4,56.4,56.3,56.3


In [87]:
print(edu_data.shape)
edu_data.head()

(32, 74)


,Наименование,Брой на студентите в колежи и университети на 1000 души,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73
0,Година,2012.0,2013.0,2014.0,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0
1,Благоевград,41.3,42.9,41.6,38.5,34.9,32.4,30.8,29.6,29.4,...,88.2,88.8,89.2,88.9,88.3,89.4,89.6,92.6,93.8,95.3
2,Бургас,23.6,23.4,21.7,19.8,17.3,15.9,14.4,14.8,15.1,...,84.0,81.5,80.1,79.3,80.0,78.8,79.0,88.4,88.1,89.0
3,Варна,69.3,67.2,65.2,63.8,56.9,52.9,51.0,50.9,50.1,...,81.2,79.8,78.9,79.2,80.1,80.4,81.2,90.3,90.5,90.5
4,Велико Търново,109.3,107.3,108.3,97.4,89.0,77.4,69.5,67.2,68.1,...,79.6,76.4,75.4,76.6,78.1,79.1,80.6,90.6,92.0,92.8


### 2.1 Data Cleaning: Addressing Hierarchical Headers

**Problem Identification**
During the initial data exploration of the socio-economic datasets (Income, Labor, Education), we encountered a structural inconsistency common to data converted from Excel. The headers were originally merged cells in the source files, which resulted in a "split-header" format when converted to CSV. Specifically, top-level categories were associated with multiple years, but the CSV export left the columns for the subsequent years empty (represented as `Unnamed` in Pandas).

**Methodology for Resolution**
To preserve the integrity of the data and enable effective merging, we implemented a robust cleaning pipeline:
1. **MultiIndex Loading:** We instructed Pandas to parse the first two rows as a hierarchical header (`header=[0, 1]`), correctly identifying the relationship between categories and specific years.
2. **Forward-Filling Categories:** We utilized a forward-fill (`ffill()`) strategy on the category-level header to propagate the label across the relevant years, ensuring that every column is fully identified by its category and its corresponding year.

In [97]:
def clean_macro_data(file_path):
    # 1. Load the file, keeping the first two rows as the header
    raw_data = pd.read_csv(file_path, header=[0, 1])
    
    # 2. The first column name is currently a tuple like ('Наименование', 'Година')
    # Clean that up to be just 'District'
    raw_data.columns.values[0] = ('District', 'District')
    
    # 3. Forward fill the top-level header (the Category names)
    new_cols = pd.MultiIndex.from_tuples(
        [(c[0] if not 'Unnamed' in c[0] else None, c[1]) for c in raw_data.columns]
    )
    
    # Forward fill the categories
    category_cols = pd.Series(new_cols.get_level_values(0)).ffill()
    year_cols = new_cols.get_level_values(1)
    
    raw_data.columns = pd.MultiIndex.from_arrays([category_cols, year_cols])
    
    return raw_data

# Apply the fix
income_data = clean_macro_data('data/E1_25.csv')
labor_data = clean_macro_data('data/E2_25.csv')
edu_data = clean_macro_data('data/S2_25.csv')

# Verification: Display the first few rows and the column structure
print("Cleaned Data Header Structure:")
display(income_data.head(3))

Cleaned Data Header Structure:


District БВП на човек от населението, лв.                             \
      District                             2012     2013     2014     2015   
0  Благоевград                           7533.0   7660.0   7659.0   8013.0   
1       Бургас                           9881.0  10101.0   9449.0  10943.0   
2        Варна                          11631.0  11543.0  12576.0  13285.0   

                                                ...  \
      2016     2017     2018     2019     2020  ...   
0   8449.0   9069.0   9989.0  10627.0  11179.0  ...   
1  12174.0  13283.0  13460.0  14483.0  12351.0  ...   
2  13756.0  14992.0  16537.0  17545.0  17379.0  ...   

  Коефициент на Джини на подоходно неравенство                    \
                                          2021  2022  2023  2024   
0                                         34.1  32.1  27.6  28.6   
1                                         32.6  38.0  29.7  31.7   
2                                         35.9  33.5  33.1  37.1   

  Относителен дял на населението, живеещо под линията на бедността за страната, %  \
                                                                             2019   
0                                               23.9                                
1                                               20.0                                
2                                               18.4                                

                                 
   2020  2021  2022  2023  2024  
0  25.1  19.1  22.9  21.6  19.5  
1  26.5  24.6  22.1  21.0  23.0  
2  22.9  17.3  14.2  14.6  15.2  

[3 rows x 55 columns]